# Experiment: WOP Python vs C++ Comparison

This notebook runs both implementations and compares estimator values and runtime.

Expected workspace layout:
- `external_wop` (Python package)
- `external_wop_cpp` (C++ project)


In [1]:
from __future__ import annotations

import json
import shutil
import statistics
import subprocess
import sys
import time
from pathlib import Path


def locate_workspace_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / "external_wop").exists() and (candidate / "external_wop_cpp").exists():
            return candidate
    raise FileNotFoundError("Could not find workspace root with external_wop and external_wop_cpp")


WORKSPACE_ROOT = locate_workspace_root(Path.cwd())
PY_ROOT = WORKSPACE_ROOT / "external_wop"
CPP_ROOT = WORKSPACE_ROOT / "external_wop_cpp"

print(f"WORKSPACE_ROOT = {WORKSPACE_ROOT}")
print(f"PY_ROOT       = {PY_ROOT}")
print(f"CPP_ROOT      = {CPP_ROOT}")


WORKSPACE_ROOT = c:\Users\Master\Desktop\Data\Диплом
PY_ROOT       = c:\Users\Master\Desktop\Data\Диплом\external_wop
CPP_ROOT      = c:\Users\Master\Desktop\Data\Диплом\external_wop_cpp


## Prepare C++ CLI

This cell configures and builds `wop_cli` in `external_wop_cpp/build`.


In [2]:
cmake_exe = shutil.which("cmake")
if cmake_exe is None:
    cmake_candidates = [
        Path("C:/Program Files/CMake/bin/cmake.exe"),
        Path("C:/Program Files (x86)/CMake/bin/cmake.exe"),
    ]
    cmake_exe = next((str(p) for p in cmake_candidates if p.exists()), None)

if cmake_exe is None:
    raise FileNotFoundError(
        "CMake is not available in PATH. Install CMake and restart Jupyter, "
        "or add cmake.exe to PATH."
    )

print(f"Using CMake: {cmake_exe}")

subprocess.run([
    cmake_exe, "-S", ".", "-B", "build", "-DCMAKE_BUILD_TYPE=Release"
], cwd=CPP_ROOT, check=True)

subprocess.run([
    cmake_exe, "--build", "build", "--config", "Release"
], cwd=CPP_ROOT, check=True)

cpp_candidates = [
    CPP_ROOT / "build" / "Release" / "wop_cli.exe",
    CPP_ROOT / "build" / "Debug" / "wop_cli.exe",
    CPP_ROOT / "build" / "wop_cli.exe",
    CPP_ROOT / "build" / "wop_cli",
]

CPP_CLI = next((p for p in cpp_candidates if p.exists()), None)
if CPP_CLI is None:
    raise FileNotFoundError("wop_cli was not found in external_wop_cpp/build")

CPP_CLI


Using CMake: C:\Program Files\CMake\bin\cmake.EXE


WindowsPath('c:/Users/Master/Desktop/Data/Диплом/external_wop_cpp/build/Release/wop_cli.exe')

## Comparison Helpers

Python run uses `python -m wop.cli` from `external_wop`.
C++ run uses `external_wop_cpp` CLI with `--json`.


In [3]:
INT_KEYS = {"n_total", "n_truncated"}
FLOAT_KEYS = {"J", "exact", "abs_error", "S2", "eps", "mean_steps"}


def parse_python_cli(raw: str) -> dict[str, float | int]:
    parsed: dict[str, float | int] = {}
    for line in raw.splitlines():
        if ":" not in line:
            continue
        key, value = line.split(":", 1)
        key = key.strip()
        value = value.strip()
        if key in INT_KEYS:
            parsed[key] = int(float(value))
        elif key in FLOAT_KEYS:
            parsed[key] = float(value)
    return parsed


def run_python_once(n_paths: int, seed: int, x0: tuple[float, float, float] = (3.0, 0.0, 0.0)) -> dict[str, float | int]:
    cmd = [
        sys.executable,
        "-m",
        "wop.cli",
        "--example",
        "box",
        "--x0",
        f"{x0[0]} {x0[1]} {x0[2]}",
        "--n",
        str(int(n_paths)),
        "--seed",
        str(int(seed)),
    ]
    t0 = time.perf_counter()
    out = subprocess.check_output(cmd, cwd=PY_ROOT, text=True)
    elapsed = time.perf_counter() - t0
    parsed = parse_python_cli(out)
    parsed["runtime_sec"] = elapsed
    return parsed


def run_cpp_once(n_paths: int, seed: int, x0: tuple[float, float, float] = (3.0, 0.0, 0.0)) -> dict[str, float | int]:
    cmd = [
        str(CPP_CLI),
        "--example",
        "box",
        "--x0",
        f"{x0[0]} {x0[1]} {x0[2]}",
        "--n",
        str(int(n_paths)),
        "--seed",
        str(int(seed)),
        "--json",
    ]
    t0 = time.perf_counter()
    out = subprocess.check_output(cmd, cwd=CPP_ROOT, text=True)
    elapsed = time.perf_counter() - t0
    parsed = json.loads(out)
    parsed["runtime_sec"] = elapsed
    return parsed


## Run Comparative Tests

Adjust `N_VALUES` and `SEEDS` if needed. For very large `N` (for example `1_000_000`), run separate single-case cells instead of the full matrix.


In [4]:
N_VALUES = [100, 1000, 10000, 100000, 1000000]
SEEDS = [314159]

rows: list[dict[str, float | int]] = []

for n_paths in N_VALUES:
    for seed in SEEDS:
        py = run_python_once(n_paths=n_paths, seed=seed)
        cpp = run_cpp_once(n_paths=n_paths, seed=seed)
        rows.append({
            "n_paths": n_paths,
            "seed": seed,
            "J_py": float(py["J"]),
            "J_cpp": float(cpp["J"]),
            "abs_delta_J": abs(float(py["J"]) - float(cpp["J"])),
            "eps_py": float(py["eps"]),
            "eps_cpp": float(cpp["eps"]),
            "runtime_py_sec": float(py["runtime_sec"]),
            "runtime_cpp_sec": float(cpp["runtime_sec"]),
            "speedup_cpp_vs_py": float(py["runtime_sec"]) / float(cpp["runtime_sec"]),
        })

rows[:3]


CalledProcessError: Command '['c:\\Users\\Master\\Desktop\\Data\\Диплом\\.venv\\Scripts\\python.exe', '-m', 'wop.cli', '--example', 'box', '--x0', '3.0 0.0 0.0', '--n', '100', '--seed', '314159']' returned non-zero exit status 1.

In [ ]:
try:
    import pandas as pd
    display(pd.DataFrame(rows))
except Exception as exc:
    print("pandas is optional; raw rows shown above")
    print(exc)


,n_paths,seed,J_py,J_cpp,abs_delta_J,eps_py,eps_cpp,runtime_py_sec,runtime_cpp_sec,speedup_cpp_vs_py
0,100,314159,0.371631,0.385107,0.013477,0.128373,0.120312,0.207059,0.002767,74.824723
1,100,271828,0.366011,0.314609,0.051402,0.124703,0.127462,0.204827,0.003049,67.181260
2,100,161803,0.329474,0.417855,0.088381,0.126270,0.131358,0.169218,0.002799,60.450407
3,1000,314159,0.370602,0.358072,0.012530,0.040746,0.039779,0.774717,0.006949,111.480111
4,1000,271828,0.340426,0.335393,0.005034,0.039019,0.039243,0.799849,0.009751,82.028688
5,1000,161803,0.352031,0.352073,0.000042,0.040075,0.039149,0.834182,0.009539,87.447202
6,10000,314159,0.351988,0.349303,0.002686,0.012544,0.012535,6.903610,0.078696,87.724927
7,10000,271828,0.356479,0.347755,0.008724,0.012633,0.012551,6.806638,0.056944,119.531925
8,10000,161803,0.353234,0.353659,0.000425,0.012588,0.012520,6.872573,0.063428,108.351853
9,100000,314159,0.353599,0.355754,0.002155,0.003986,0.003986,67.772205,0.539447,125.632707
